# To run syncode need to set runtime version to different version that is NOT latest (Python 13)


In [1]:
import sys
print(sys.version)

!pip install syncode
!pip install transformers==4.53.2

from syncode import Syncode

3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


In [65]:
# Pass in a grammar as a string of rules in EBNF format

# grammar = """ start: "[" operands "," operator "]"

#               operands: "[" number "," number "]"
#               number: /-?[0-9]+(\\.[0-9]*)/

#               operator: "\\"*\\"" | "\\"+\\"" | "\\"-\\"" | "\\"/\\""
# """

grammar = """ start: "[" operands "," operator "]"

              operands: "[" number "," number "]"
              number: /-?[0-9]+(\\.[0-9]*)/

              operator: "\\"plus\\"" | "\\"minus\\"" | "\\"divided by\\"" | "\\"multiplied by\\""
"""

In [66]:
few_shot = """ Extract the two floating point operands in their original order and the single operator from the query.

               Rules:
                1. Identify the first and second operands independently.
                2. Preserve the sign of each operand.
                3. If there is a negative sign immediately before a number, the negative sign belongs to that number.
                4. Both operands can be negative even when the operation is division.
                5. Do not remove or change negative signs based on the mathematical operation.
                6. Return only the operands and the operator in a list.
                7. Do not calculate the answer or add any verbose explanation.

               Examples:

               Query: What is 327. multiplied by 11.0?
               Output: [[327.0, 11.0], "multiplied by"]

               Query: What is 45.1 plus 23.54?
               Output: [[45.1, 23.54], "plus"]

               Query: What is -120.4 divided by 4.0?
               Output: [[-120.4, 4.0], "divided by"]

               Query: What is -50.5 minus -12.2?
               Output: [[-50.5, -12.2], "minus"]

               Query: What is -3.0 divided by -1.0?
               Output: [[-3.0, -1.0], "divided by"]

               Query: What is -4 divided by -2?
               Output: [[-4.0, -2.0], "divided by"]
"""

# few_shot = """ Extract the two floating point operands in their original order and the single operator from the query.

#                Rules:
#                 1. Identify the first and second operands independently.
#                 2. Preserve the sign of each operand.
#                 3. If there is a negative sign immediately before a number, the negative sign belongs to that number.
#                 4. Both operands can be negative even when the operation is division.
#                 5. Do not remove or change negative signs based on the mathematical operation.
#                 6. Return only the operands and the operator in a list.
#                 7. Do not calculate the answer or add any verbose explanation.


#                Examples:

#                Query: What is 327. multiplied by 11.0?
#                Output: [[327.0, 11.0], "*"]

#                Query: What is 45.1 plus 23.54?
#                Output: [[45.1, 23.54], "+"]

#                Query: What is -120.4 divided by 4.0?
#                Output: [[-120.4, 4.0], "/"]

#                Query: What is -50.5 minus -12.2?
#                Output: [[-50.5, -12.2], "-"]

#                Query: What is -3.0 divided by -1.0?
#                Output: [[-3.0, -1.0], "/"]

#                Query: What is -4 divided by -2?
#                Output: [[-4.0, -2.0], "/"]
# """



In [67]:
model_name = "microsoft/phi-2"

# Load the Syncode augmented model
syn_llm = Syncode(model=model_name, grammar=grammar, parse_output_only=True)

[2026-10-28 01:44:42,455-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16
[2026-10-28 01:44:42,643-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [68]:
import ast
import operator

operators = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv
}

operators_text = {
    "plus" : "+",
    "minus": "-",
    "multiplied by": "*",
    "divided by": "/"
}

def run_syncode(textual_query):
  textual_query = textual_query.strip()

  prompt = few_shot + "\nQuery: " + textual_query + "\nOutput: "
  output = syn_llm.infer(prompt)[0]

  operands, operator = ast.literal_eval(output)
  operands = [float(o) for o in operands]

  # answer = float(operators[operator](*operands))
  answer = float(operators[operators_text[operator]](*operands))

  # return operands, operator, answer
  return operands, operators_text[operator], answer


In [69]:
# checking

q1 = "What is 327. multiplied by 11.0?"

q2 = "What is 45.1 plus 23.54?"

q3 = "What is -120.4 divided by 4.0?"

q4 = "What is -50.5 minus -12.2?"

q5 = "What is -160.0 divided by -10.0?"

q6 = "What is 3.0 multiplied by 12.0?"

q7 = "What is 3 plus 4?"

q8 = "What is 5 minus 6?"

q9 = "What is 3.0 plus 4.0?"

q10 = "What is 5.0 minus 6.0?"

q_list = [q1, q2, q3, q4, q5, q6, q7, q8, q9, q10]

for q in q_list:
  print(q)
  print(run_syncode(q))

  print()

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is 327. multiplied by 11.0?
([327.0, 11.0], '*', 3597.0)

What is 45.1 plus 23.54?
([45.1, 23.54], '+', 68.64)

What is -120.4 divided by 4.0?
([-120.4, 4.0], '/', -30.1)

What is -50.5 minus -12.2?
([-50.5, -12.2], '-', -38.3)

What is -160.0 divided by -10.0?
([-160.0, 10.0], '/', -16.0)

What is 3.0 multiplied by 12.0?
([3.0, 12.0], '*', 36.0)

What is 3 plus 4?
([3.0, 4.0], '+', 7.0)

What is 5 minus 6?
([5.0, 6.0], '-', -1.0)

What is 3.0 plus 4.0?
([3.0, 4.0], '+', 7.0)

What is 5.0 minus 6.0?
([5.0, 6.0], '-', -1.0)



In [70]:
run_syncode("What is -23 divided by -46?")

([-23.0, -46.0], '/', 0.5)

In [71]:
run_syncode("What is -23.0 divided by -46.0?")

([-23.0, -46.0], '/', 0.5)

In [75]:
run_syncode("What is -1600.0 divided by -10.0?") #ugh whatever

([-1600.0, 10.0], '/', -160.0)